# DriveWise: Metadata-Aware Automotive RAG Assistant
## Final Project

A conversational AI system that helps users understand car brochures by answering natural language queries using brochure-grounded information. Uses RAG with metadata filtering, structured chunking, re-ranking, and source attribution.

**Pipeline:** Brochure Loading → Structured Chunking → Embedding → ChromaDB (with metadata) → Query → Metadata Filter → Vector Search → Re-rank → Context Control → Answer Generation → Source Attribution


In [ ]:
import chromadb
import numpy as np
import time
import json
import logging
from datetime import datetime
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger("DriveWise")

print("All imports loaded")


## Part 1 -- Brochure Data (Document Ingestion)

In a real scenario these would be extracted from actual PDF brochures downloaded from car company websites. Here we have sample brochure content for Hyundai, Tata, and Maruti cars structured by sections like engine, safety, mileage etc.


In [ ]:
# sample brochure data structured by brand > model > section
# each section has text content and page number (metadata)

brochures = {
    "Hyundai": {
        "Creta": {
            "overview": {
                "text": "The Hyundai Creta is a compact SUV designed for Indian roads. It comes with a bold front grille, sharp LED headlamps, and connected LED tail lamps. The Creta is available in multiple variants including E, EX, S, SX, and SX(O). It offers a premium driving experience with advanced technology and safety features.",
                "page": 1
            },
            "engine_performance": {
                "text": "The Hyundai Creta is available with three engine options. The 1.5L naturally aspirated petrol engine produces 115 PS of power and 144 Nm of torque, paired with a 6-speed manual or IVT automatic transmission. The 1.5L diesel engine delivers 116 PS of power and 250 Nm of torque with a 6-speed manual or 6-speed automatic transmission. The 1.4L turbo petrol engine generates 140 PS and 242 Nm of torque, mated to a 7-speed DCT automatic.",
                "page": 2
            },
            "mileage_fuel": {
                "text": "The Hyundai Creta offers competitive fuel efficiency across all variants. The 1.5L petrol manual delivers 16.8 kmpl while the IVT automatic gives 16.5 kmpl. The 1.5L diesel manual offers an impressive 21.4 kmpl and the automatic variant returns 18.5 kmpl. The 1.4L turbo petrol with DCT delivers 16.6 kmpl. The fuel tank capacity is 50 litres for all variants.",
                "page": 3
            },
            "safety": {
                "text": "The Hyundai Creta comes loaded with safety features. It has 6 airbags across all variants, ABS with EBD, Electronic Stability Control (ESC), Hill Assist Control (HAC), Vehicle Stability Management (VSM), rear parking sensors, and a reverse camera. The higher variants also get ADAS Level 2 features including Forward Collision Avoidance, Lane Keeping Assist, Smart Cruise Control, and Blind Spot Collision Warning. The Creta has received a 5-star safety rating from Global NCAP.",
                "page": 4
            },
            "dimensions": {
                "text": "The Hyundai Creta measures 4330 mm in length, 1790 mm in width, and 1635 mm in height. The wheelbase is 2610 mm which provides ample cabin space. The ground clearance is 190 mm. The boot space is 433 litres. The kerb weight ranges from 1210 kg to 1380 kg depending on the variant.",
                "page": 5
            },
            "interior_comfort": {
                "text": "The Hyundai Creta interior features a 10.25-inch touchscreen infotainment system with wireless Android Auto and Apple CarPlay. It has a fully digital 10.25-inch instrument cluster, Bose premium 8-speaker sound system, ventilated front seats, powered driver seat, panoramic sunroof, air purifier, rear AC vents, wireless phone charger, and ambient lighting.",
                "page": 6
            },
            "price_variants": {
                "text": "The Hyundai Creta price starts from Rs 11.0 lakh for the base E petrol variant and goes up to Rs 20.15 lakh for the top SX(O) turbo DCT variant (ex-showroom). The diesel variants range from Rs 12.5 lakh to Rs 19.99 lakh. The most popular variant is the SX variant which offers the best value for money.",
                "page": 7
            }
        },
        "Venue": {
            "overview": {
                "text": "The Hyundai Venue is a sub-compact SUV designed for urban city driving. It features a sporty and youthful design with a cascading front grille, LED projector headlamps, and dual-tone roof options. Available in E, S, SX, and SX(O) variants.",
                "page": 1
            },
            "engine_performance": {
                "text": "The Hyundai Venue comes with two engine options. The 1.2L Kappa petrol engine produces 83 PS and 114 Nm of torque paired with a 5-speed manual transmission. The 1.0L turbo GDi petrol engine delivers 120 PS and 172 Nm of torque mated to a 7-speed DCT automatic or 6-speed iMT transmission.",
                "page": 2
            },
            "mileage_fuel": {
                "text": "The Hyundai Venue fuel efficiency varies by engine. The 1.2L petrol manual delivers 17.5 kmpl. The 1.0L turbo with iMT gives 18.1 kmpl while the turbo DCT automatic returns 17.2 kmpl. The fuel tank capacity is 45 litres.",
                "page": 3
            },
            "safety": {
                "text": "The Hyundai Venue safety features include 6 airbags, ABS with EBD, rear parking sensors with camera, Hill Assist Control, Electronic Stability Control, and ISOFIX child seat mounts. The higher variants also get TPMS.",
                "page": 4
            },
            "dimensions": {
                "text": "The Hyundai Venue is 3995 mm long, 1770 mm wide, and 1617 mm tall. The wheelbase is 2500 mm and the ground clearance is 195 mm. The boot space is 350 litres. The turning radius is 5.2 metres.",
                "page": 5
            },
            "price_variants": {
                "text": "The Hyundai Venue starts from Rs 7.94 lakh for the base E variant and goes up to Rs 13.48 lakh for the top SX(O) turbo DCT variant (ex-showroom).",
                "page": 6
            }
        }
    },
    "Tata": {
        "Nexon": {
            "overview": {
                "text": "The Tata Nexon is a sub-compact SUV and one of India's highest-selling SUVs. It was the first car in India to receive a 5-star Global NCAP safety rating. Available in Smart, Pure, Creative, and Fearless variants.",
                "page": 1
            },
            "engine_performance": {
                "text": "The Tata Nexon comes with two engine options. The 1.2L turbocharged Revotron petrol engine produces 120 PS and 170 Nm of torque. The 1.5L turbocharged Revotorq diesel engine delivers 116 PS and 260 Nm of torque. Both engines are available with a 6-speed manual or 6-speed AMT automatic transmission.",
                "page": 2
            },
            "mileage_fuel": {
                "text": "The Tata Nexon petrol manual delivers 17.4 kmpl while the AMT gives 17.2 kmpl. The diesel manual offers 23.2 kmpl and the diesel AMT gives 22.1 kmpl. The fuel tank is 44 litres.",
                "page": 3
            },
            "safety": {
                "text": "The Tata Nexon has a 5-star Global NCAP safety rating. It comes with 6 airbags across all variants, ABS with EBD, Corner Stability Control, Electronic Stability Program, Hill Hold Control, ISOFIX child seat mounts, all four disc brakes, and rear parking sensors with camera.",
                "page": 4
            },
            "dimensions": {
                "text": "The Tata Nexon measures 3994 mm in length, 1811 mm in width, and 1607 mm in height. The wheelbase is 2498 mm and ground clearance is 209 mm which is the best in segment. Boot space is 350 litres.",
                "page": 5
            },
            "price_variants": {
                "text": "The Tata Nexon price starts from Rs 8.10 lakh for the Smart petrol variant and goes up to Rs 15.50 lakh for the top Fearless Plus diesel AMT variant (ex-showroom). The most popular variant is Creative Plus at around Rs 12.0 lakh.",
                "page": 6
            }
        }
    },
    "Maruti Suzuki": {
        "Brezza": {
            "overview": {
                "text": "The Maruti Suzuki Brezza is a sub-compact SUV known for its reliability, low maintenance costs, and strong resale value. Available in LXi, VXi, ZXi, and ZXi+ variants.",
                "page": 1
            },
            "engine_performance": {
                "text": "The Maruti Brezza is powered by a 1.5L K15C Dual Jet Dual VVT petrol engine with Smart Hybrid technology producing 103 PS and 137 Nm of torque. It is available with a 5-speed manual or 6-speed torque converter automatic transmission.",
                "page": 2
            },
            "mileage_fuel": {
                "text": "The Maruti Brezza manual delivers 20.15 kmpl while the automatic gives 19.80 kmpl. The fuel tank capacity is 48 litres. CNG variant is also available delivering 25.0 km/kg.",
                "page": 3
            },
            "safety": {
                "text": "The Maruti Brezza comes with 6 airbags, ABS with EBD, ESP with Hill Hold Assist, rear parking sensors with camera, ISOFIX child seat mounts, and high-speed alert system. Top variants also get a 360-degree camera.",
                "page": 4
            },
            "dimensions": {
                "text": "The Maruti Brezza measures 3995 mm in length, 1790 mm in width, and 1640 mm in height. The wheelbase is 2500 mm and ground clearance is 198 mm. Boot space is 328 litres.",
                "page": 5
            },
            "price_variants": {
                "text": "The Maruti Brezza starts from Rs 8.34 lakh for the LXi manual and goes up to Rs 14.14 lakh for the ZXi+ automatic variant (ex-showroom). Maruti's service costs are among the lowest in the industry.",
                "page": 6
            }
        }
    }
}

# count total chunks
total = sum(len(sections) for models in brochures.values() for sections in models.values())
print(f"Loaded brochure data for {len(brochures)} brands")
print(f"Total models: {sum(len(m) for m in brochures.values())}")
print(f"Total brochure sections: {total}")


## Part 2 -- Embedding Model & ChromaDB Setup

Loading the embedding model and setting up ChromaDB as our vector store. ChromaDB supports metadata filtering out of the box which is exactly what we need.


In [ ]:
# load embedding model
print("Loading embedding model...")
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Model loaded: all-MiniLM-L6-v2 (384 dimensions)")

# setup chromadb
client = chromadb.Client()
collection = client.get_or_create_collection(
    name="car_brochures",
    metadata={"hnsw:space": "cosine"}
)
print("ChromaDB collection created")


## Part 3 -- Structured Chunking & Ingestion with Metadata

Instead of splitting text randomly, our chunks follow the brochure structure (engine, safety, mileage etc). Each chunk gets tagged with brand, model, section name, and page number as metadata.


In [ ]:
# ingest all brochure data into chromadb with metadata
documents = []
metadatas = []
ids = []
idx = 0

for brand, models in brochures.items():
    for model, sections in models.items():
        for section_name, section_data in sections.items():
            documents.append(section_data["text"])
            metadatas.append({
                "brand": brand,
                "model": model,
                "section": section_name,
                "page": section_data["page"]
            })
            ids.append(f"chunk_{idx}")
            idx += 1

# embed everything
embeddings = embed_model.encode(documents).tolist()

# add to chromadb
collection.add(
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas,
    ids=ids
)

print(f"Ingested {idx} chunks into ChromaDB")
print(f"\nSample metadata:")
for m in metadatas[:3]:
    print(f"  {m}")


## Part 4 -- Retrieval with Metadata Filtering

When the user picks a brand and model, we filter the vector store using metadata BEFORE doing similarity search. This way we only search within the relevant car's brochure.


In [ ]:
def retrieve(query, brand=None, model=None, top_k=5):
    # build metadata filter based on user selection
    where_filter = None
    if brand and model:
        where_filter = {"$and": [
            {"brand": {"$eq": brand}},
            {"model": {"$eq": model}}
        ]}
    elif brand:
        where_filter = {"brand": {"$eq": brand}}

    query_embedding = embed_model.encode([query]).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k,
        where=where_filter,
        include=["documents", "metadatas", "distances"]
    )

    retrieved = []
    if results["documents"] and results["documents"][0]:
        for i in range(len(results["documents"][0])):
            retrieved.append({
                "text": results["documents"][0][i],
                "metadata": results["metadatas"][0][i],
                "distance": results["distances"][0][i],
                "relevance": 1 - results["distances"][0][i]
            })
    return retrieved

# test retrieval with metadata filter
print("Testing retrieval with metadata filter: brand=Hyundai, model=Creta")
results = retrieve("what is the mileage?", brand="Hyundai", model="Creta", top_k=3)

for r in results:
    print(f"\n  Section: {r['metadata']['section']}, Page: {r['metadata']['page']}")
    print(f"  Relevance: {r['relevance']:.4f}")
    print(f"  Text: {r['text'][:100]}...")


## Part 5 -- Re-Ranking

After initial retrieval, we re-score each chunk against the query using cosine similarity to make sure the best chunks are on top.


In [ ]:
def rerank(query, chunks, top_k=3):
    if not chunks:
        return []

    query_emb = embed_model.encode([query])
    chunk_embs = embed_model.encode([c["text"] for c in chunks])
    scores = cosine_similarity(query_emb, chunk_embs)[0]

    for i, chunk in enumerate(chunks):
        chunk["rerank_score"] = float(scores[i])

    reranked = sorted(chunks, key=lambda x: x["rerank_score"], reverse=True)
    return reranked[:top_k]

# test reranking
query = "does Creta have a sunroof?"
retrieved = retrieve(query, brand="Hyundai", model="Creta", top_k=5)
reranked = rerank(query, retrieved, top_k=3)

print(f"Query: '{query}'")
print(f"\nAfter re-ranking (top 3):")
for r in reranked:
    print(f"  {r['metadata']['section']} -> score: {r['rerank_score']:.4f}")


## Part 6 -- Context Window Control & Answer Generation

We limit the context to avoid sending too much text. Then we extract the most relevant sentences from the context to form the answer.


In [ ]:
def control_context(chunks, max_chars=2000):
    selected = []
    total = 0
    for c in chunks:
        if total + len(c["text"]) <= max_chars:
            selected.append(c)
            total += len(c["text"])
    return selected

def generate_answer(query, chunks):
    if not chunks:
        return "I couldn't find relevant information for this question."

    context = " ".join([c["text"] for c in chunks])

    # split into sentences and score them against the query
    sentences = [s.strip() for s in context.replace(". ", ".\n").split("\n") if len(s.strip()) > 15]

    if not sentences:
        return context[:500]

    query_emb = embed_model.encode([query])
    sent_embs = embed_model.encode(sentences)
    scores = cosine_similarity(query_emb, sent_embs)[0]

    # pick top 4 sentences and reorder them to match original flow
    scored = sorted(zip(sentences, scores), key=lambda x: x[1], reverse=True)
    top_sents = set(s for s, _ in scored[:4])
    ordered = [s for s in sentences if s in top_sents]

    return " ".join(ordered)

# test it
query = "what is the mileage of Hyundai Creta?"
retrieved = retrieve(query, brand="Hyundai", model="Creta")
reranked = rerank(query, retrieved)
final_chunks = control_context(reranked)
answer = generate_answer(query, final_chunks)

print(f"Q: {query}")
print(f"A: {answer}")


## Part 7 -- Full RAG Pipeline with Source Attribution

Putting everything together: retrieve -> rerank -> context control -> generate -> attribute sources


In [ ]:
def ask(query, brand=None, model=None):
    start = time.time()

    # retrieve with metadata filter
    retrieved = retrieve(query, brand, model, top_k=5)
    if not retrieved:
        return {"answer": "No data found for this car.", "sources": [], "time": 0}

    # rerank
    reranked = rerank(query, retrieved, top_k=3)

    # context window control
    final = control_context(reranked)

    # generate answer
    answer = generate_answer(query, final)

    # source attribution
    sources = []
    for c in final:
        sources.append({
            "brand": c["metadata"]["brand"],
            "model": c["metadata"]["model"],
            "section": c["metadata"]["section"],
            "page": c["metadata"]["page"],
            "relevance": round(c["rerank_score"], 4)
        })

    elapsed = round(time.time() - start, 3)
    return {"answer": answer, "sources": sources, "time": elapsed}


## Part 8 -- Demo Q&A

Running the full pipeline on different car questions


In [ ]:
# test questions for different cars
test_cases = [
    ("Hyundai", "Creta", "What is the mileage of Hyundai Creta?"),
    ("Hyundai", "Creta", "How many airbags does Creta have?"),
    ("Hyundai", "Creta", "What is the price range?"),
    ("Hyundai", "Venue", "What engine options does Venue have?"),
    ("Tata", "Nexon", "What is the safety rating of Nexon?"),
    ("Tata", "Nexon", "What is the boot space?"),
    ("Maruti Suzuki", "Brezza", "What is the fuel efficiency of Brezza?"),
    (None, None, "Which car has the best mileage?"),
]

print("=" * 70)
print("DriveWise Q&A Demo")
print("=" * 70)

for brand, model, question in test_cases:
    result = ask(question, brand, model)
    car_label = f"{brand} {model}" if brand else "All Cars"

    print(f"\n[{car_label}] Q: {question}")
    print(f"A: {result['answer'][:200]}...")
    print(f"Sources: {[f\"{s['section']} (p{s['page']}, {s['relevance']:.0%})\" for s in result['sources']]}")
    print(f"Time: {result['time']}s")
    print("-" * 70)


## Part 9 -- Evaluation Metrics


In [ ]:
def evaluate(query, answer, chunks):
    if not chunks or not answer:
        return {"faithfulness": 0, "context_relevance": 0}

    # faithfulness: how much of the answer words come from context
    context_text = " ".join([c["text"] for c in chunks]).lower()
    answer_words = answer.lower().split()
    grounded = sum(1 for w in answer_words if w in context_text and len(w) > 3)
    faithfulness = grounded / max(len(answer_words), 1)

    # context relevance: average rerank score
    avg_rel = np.mean([c.get("rerank_score", 0) for c in chunks])

    return {
        "faithfulness": round(faithfulness, 3),
        "context_relevance": round(float(avg_rel), 3)
    }

# evaluate each test case
print("\n--- Evaluation Results ---\n")
eval_results = []

for brand, model, question in test_cases:
    retrieved = retrieve(question, brand, model)
    reranked = rerank(question, retrieved)
    final = control_context(reranked)
    answer = generate_answer(question, final)
    ev = evaluate(question, answer, final)
    eval_results.append(ev)

    car_label = f"{brand} {model}" if brand else "All Cars"
    print(f"[{car_label}] {question[:40]}...")
    print(f"  Faithfulness: {ev['faithfulness']:.1%} | Context Relevance: {ev['context_relevance']:.1%}")

avg_faith = np.mean([e["faithfulness"] for e in eval_results])
avg_rel = np.mean([e["context_relevance"] for e in eval_results])
print(f"\nOverall Faithfulness: {avg_faith:.1%}")
print(f"Overall Context Relevance: {avg_rel:.1%}")


## Part 10 -- Visualizations


In [ ]:
# 10a: Embedding space of all brochure chunks
all_embeddings = np.array(embed_model.encode(documents))
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
emb_2d = pca.fit_transform(all_embeddings)

# color by brand
brand_colors = {"Hyundai": "#1a73e8", "Tata": "#0f4c81", "Maruti Suzuki": "#e53935"}
colors = [brand_colors.get(m["brand"], "gray") for m in metadatas]

plt.figure(figsize=(10, 6))
for brand, color in brand_colors.items():
    mask = [m["brand"] == brand for m in metadatas]
    pts = emb_2d[mask]
    if len(pts) > 0:
        plt.scatter(pts[:, 0], pts[:, 1], c=color, label=brand, s=80, alpha=0.7, edgecolors='black')

plt.title("Brochure Chunks in Embedding Space (PCA)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# 10b: Evaluation bar chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

labels = [f"Q{i+1}" for i in range(len(eval_results))]
faith_scores = [e["faithfulness"] for e in eval_results]
rel_scores = [e["context_relevance"] for e in eval_results]

ax1.bar(labels, faith_scores, color='#4CAF50', edgecolor='black', alpha=0.8)
ax1.set_ylabel("Score")
ax1.set_title("Faithfulness per Query")
ax1.set_ylim(0, 1)
ax1.axhline(y=avg_faith, color='red', linestyle='--', label=f'Avg: {avg_faith:.2f}')
ax1.legend()

ax2.bar(labels, rel_scores, color='#2196F3', edgecolor='black', alpha=0.8)
ax2.set_ylabel("Score")
ax2.set_title("Context Relevance per Query")
ax2.set_ylim(0, 1)
ax2.axhline(y=avg_rel, color='red', linestyle='--', label=f'Avg: {avg_rel:.2f}')
ax2.legend()

plt.tight_layout()
plt.show()


In [ ]:
# 10c: chunk similarity heatmap
sim_matrix = cosine_similarity(all_embeddings)

plt.figure(figsize=(10, 8))
plt.imshow(sim_matrix, cmap='YlOrRd', interpolation='nearest')
plt.colorbar(label='Cosine Similarity')
plt.title("Brochure Chunk Similarity Heatmap")
plt.xlabel("Chunk Index")
plt.ylabel("Chunk Index")
plt.tight_layout()
plt.show()


## Part 11 -- System Architecture

```
                        DriveWise Architecture
                        
User Input ──────────────────────────────────────────────────
  │  Brand Selection                                        
  │  Model Selection                                        
  │  Natural Language Question                              
  │                                                         
  ▼                                                         
Retrieval Layer ─────────────────────────────────────────────
  │  1. Metadata Filtering (brand + model)                  
  │  2. Embedding Query (all-MiniLM-L6-v2)                  
  │  3. Vector Search (ChromaDB cosine similarity)          
  │  4. Re-Ranking (cross-similarity scoring)               
  │  5. Context Window Control (max 2000 chars)             
  │                                                         
  ▼                                                         
Generation Layer ────────────────────────────────────────────
  │  1. Extractive Answer Generation                        
  │  2. Source Attribution (section, page, relevance)       
  │                                                         
  ▼                                                         
Monitoring Layer ────────────────────────────────────────────
  │  Query Logging                                          
  │  Response Time Tracking                                 
  │  Evaluation (Faithfulness, Context Relevance)           
  │                                                         
  ▼                                                         
Output ──────────────────────────────────────────────────────
  │  Answer grounded in brochure data                       
  │  Source references (brand, model, section, page)        
  │  Evaluation scores                                      
```


## Observations

- metadata filtering is probably the most impactful part of the whole system,
  without it the retrieval pulls chunks from completely different cars and the
  answers become meaningless

- structured chunking by brochure sections (engine, safety, mileage etc) works
  much better than random character-based splitting because each chunk has a
  clear topic and the retrieval is more precise

- re-ranking helps when the initial retrieval returns borderline relevant chunks,
  it pushes the actually useful ones to the top before generating the answer

- the extractive answer approach (picking best sentences from context) keeps
  answers fully grounded in the brochure data, no hallucination risk since we
  never generate new text

- source attribution is important for trust, the user can see exactly which
  brochure section and page the answer came from

- context window control prevents sending too much text which would dilute the
  answer quality, keeping only the top 2000 chars is a good balance

- ChromaDB handles metadata filtering natively which makes the implementation
  much cleaner than manually filtering with FAISS

- evaluation metrics (faithfulness and context relevance) help identify when
  the system is struggling with certain types of questions

- for production you would use actual PDF brochures, a proper LLM for generation,
  and more sophisticated evaluation using frameworks like RAGAS


In [ ]:
print("=" * 60)
print("DriveWise Project Complete!")
print("=" * 60)
